# Silver Layer — ERP: Customer AZ12

Transforms the bronze `workspace.bronze.cust_az12` table into `workspace.silver.customer_info1` with meaningful, readable column names and cleaned data.

---

## Table of Contents — Transformations Applied

| # | Transformation | Description |
|---|---|---|
| 1 | **Column Renaming** | Renamed abbreviated columns to meaningful names (`CID` -> `customer_id`, `BDATE` -> `birth_date`, `GEN` -> `gender`) |
| 2 | **Clean Customer ID** | Trimmed whitespace and removed `NAS` prefix from `customer_id` (e.g., `NASAW00011000` -> `AW00011000`) |
| 3 | **Gender Standardization** | Trimmed and mapped all gender variants to consistent values: `M`/`M ` -> `Male`, `F`/`F ` -> `Female`, `NULL`/spaces/empty -> `Unknown` |

---

## Column Mapping (Bronze to Silver)

| Bronze Column | Silver Column | Transformation Applied |
|---|---|---|
| `CID` | `customer_id` | Renamed + Trimmed + `NAS` prefix removed |
| `BDATE` | `birth_date` | Renamed |
| `GEN` | `gender` | Renamed + Trimmed + Standardized |

---

## Gender Value Mapping

| Bronze Value(s) | Silver Value | Count |
|---|---|---|
| `Male` | `Male` | 8,608 |
| `Female` | `Female` | 8,391 |
| `NULL`, `  `, empty | `Unknown` | 1,476 |
| `M`, `M ` | `Male` | 4 |
| `F`, `F ` | `Female` | 5 |

---

## Implementation

All transformations are implemented in **Spark SQL**:
- `TRIM()` — remove leading/trailing whitespace
- `REGEXP_REPLACE()` — remove the `NAS` prefix from `customer_id`
- `CASE WHEN` — conditional mapping for gender standardization
- `CREATE OR REPLACE TABLE ... AS SELECT` — write to silver layer

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.customer_info1 AS
SELECT
    REGEXP_REPLACE(TRIM(CID), '^NAS', '') AS customer_id,
    BDATE                               AS birth_date,
    CASE TRIM(GEN)
        WHEN 'Male'   THEN 'Male'
        WHEN 'M'      THEN 'Male'
        WHEN 'Female' THEN 'Female'
        WHEN 'F'      THEN 'Female'
        ELSE 'Unknown'
    END                                 AS gender
FROM workspace.bronze.cust_az12

In [0]:
%sql
SELECT * FROM workspace.silver.customer_info1 LIMIT 10